# Exploracao de Resultados do Grid Search (AP combinado)

Este notebook recebe um `RUN_DIR` e monta tabelas e graficos para responder se cada combinacao melhorou ou piorou o desempenho.

Saidas esperadas dentro do `RUN_DIR`:
- `results_iterative.csv`
- `final_ranking.csv` (opcional)
- `best_by_model.csv` (opcional)
- `summary.json` (opcional)

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 200)
print('plotly:', HAS_PLOTLY, '| ipywidgets:', HAS_WIDGETS)
if not HAS_PLOTLY:
    print('Se desejar graficos interativos: pip install plotly')
if not HAS_WIDGETS:
    print('Se desejar filtros interativos: pip install ipywidgets')

plotly: True | ipywidgets: True


In [2]:
# Entrada principal
# RUN_DIR = Path('../data/resultados/gs_ap_combined_100k_20260517_142956')
RUN_DIR = Path('../data/resultados/gs_ap_combined_100k_20260518_231005')

if not RUN_DIR.exists():
    raise FileNotFoundError(f'RUN_DIR nao encontrado: {RUN_DIR.resolve()}')

print('RUN_DIR:', RUN_DIR.resolve())
for p in sorted(RUN_DIR.glob('*')):
    print('-', p.name)

RUN_DIR: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/resultados/gs_ap_combined_100k_20260518_231005
- best_by_model.csv
- checkpoint_state.json
- combined_scores
- config.json
- final_ranking.csv
- logs
- results_iterative.csv
- scores
- snapshots
- summary.json


In [3]:
def load_run(run_dir: Path) -> dict:
    files = {
        'results': run_dir / 'results_iterative.csv',
        'final_ranking': run_dir / 'final_ranking.csv',
        'best_by_model': run_dir / 'best_by_model.csv',
        'summary': run_dir / 'summary.json',
    }

    if not files['results'].exists():
        raise FileNotFoundError(f'Arquivo obrigatorio ausente: {files["results"]}')

    out = {'paths': files}
    out['results'] = pd.read_csv(files['results'])
    out['final_ranking'] = pd.read_csv(files['final_ranking']) if files['final_ranking'].exists() else None
    out['best_by_model'] = pd.read_csv(files['best_by_model']) if files['best_by_model'].exists() else None

    if files['summary'].exists():
        with open(files['summary'], 'r', encoding='utf-8') as f:
            out['summary'] = json.load(f)
    else:
        out['summary'] = None

    return out

bundle = load_run(RUN_DIR)
results_df = bundle['results'].copy()
print('results_iterative.csv:', results_df.shape)
results_df.head(2)

results_iterative.csv: (707, 21)


,timestamp,job_index,job_total,model,combo_id,params_json,status,ap_estabel,auc_estabel,fit_time_s_estabel,ap_lav_temp,auc_lav_temp,fit_time_s_lav_temp,ap_pecu,auc_pecu,fit_time_s_pecu,ap_combined,auc_combined,ap_individual_mean,objective_ap,total_job_time_s
0,2026-05-18T23:10:35.344098,1,425,IForest,72c1563628d4,"{""bootstrap"": false, ""max_features"": 0.1, ""max_samples"": 256, ""n_estimators"": 100}",ok,0.09186620211736037,0.579809,1.4532,0.096604,0.569773,1.4589,0.098502,0.616090,1.5734,0.093954,0.586648,0.095657,0.093954,4.8916
1,2026-05-18T23:10:40.277160,2,425,IForest,f104c7aba29e,"{""bootstrap"": true, ""max_features"": 0.1, ""max_samples"": 256, ""n_estimators"": 100}",ok,0.09195397829597152,0.579618,1.4255,0.096783,0.570943,1.4521,0.098479,0.616667,1.4764,0.094043,0.587180,0.095739,0.094043,4.7799


In [4]:
def prepare_analysis(df: pd.DataFrame) -> pd.DataFrame:
    base = df.copy()

    if 'status' in base.columns:
        base = base[base['status'].fillna('') == 'ok'].copy()

    if base.empty:
        raise ValueError('Nao ha linhas com status=ok para analisar.')

    base['job_index'] = pd.to_numeric(base.get('job_index'), errors='coerce')
    base['objective_ap'] = pd.to_numeric(base.get('objective_ap'), errors='coerce')
    base = base.dropna(subset=['job_index', 'objective_ap']).sort_values('job_index').reset_index(drop=True)

    base['delta_vs_prev_global'] = base['objective_ap'].diff().fillna(0.0)
    base['best_so_far'] = base['objective_ap'].cummax()
    base['best_so_far_before'] = base['best_so_far'].shift(1).fillna(base['objective_ap'].iloc[0])
    base['delta_vs_best_before'] = base['objective_ap'] - base['best_so_far_before']
    base['is_new_best_global'] = base['objective_ap'] >= base['best_so_far_before']

    base['delta_vs_prev_model'] = (
        base.sort_values(['model', 'job_index'])
            .groupby('model')['objective_ap']
            .diff()
            .reindex(base.index)
            .fillna(0.0)
    )

    m_sorted = base.sort_values(['model', 'job_index']).copy()
    m_sorted['best_so_far_model'] = m_sorted.groupby('model')['objective_ap'].cummax()
    m_sorted['best_so_far_model_before'] = m_sorted.groupby('model')['best_so_far_model'].shift(1)
    m_sorted['best_so_far_model_before'] = m_sorted['best_so_far_model_before'].fillna(m_sorted['objective_ap'])
    m_sorted['delta_vs_best_model_before'] = m_sorted['objective_ap'] - m_sorted['best_so_far_model_before']
    m_sorted['is_new_best_model'] = m_sorted['objective_ap'] >= m_sorted['best_so_far_model_before']

    base = base.merge(
        m_sorted[['model', 'combo_id', 'job_index', 'best_so_far_model', 'best_so_far_model_before', 'delta_vs_best_model_before', 'is_new_best_model']],
        on=['model', 'combo_id', 'job_index'],
        how='left'
    )

    base['trend_global'] = np.where(base['delta_vs_prev_global'] > 0, 'melhorou', np.where(base['delta_vs_prev_global'] < 0, 'piorou', 'estavel'))
    base['trend_model'] = np.where(base['delta_vs_prev_model'] > 0, 'melhorou', np.where(base['delta_vs_prev_model'] < 0, 'piorou', 'estavel'))
    return base

analysis_df = prepare_analysis(results_df)
print('Linhas validas para analise:', analysis_df.shape[0])
analysis_df[['job_index', 'model', 'combo_id', 'objective_ap', 'delta_vs_prev_global', 'trend_global']].head()

Linhas validas para analise: 543


,job_index,model,combo_id,objective_ap,delta_vs_prev_global,trend_global
0,1,IForest,72c1563628d4,0.093954,0.000000,estavel
1,2,IForest,f104c7aba29e,0.094043,0.000088,melhorou
2,3,IForest,ff6c39eb52f6,0.095325,0.001282,melhorou
3,4,IForest,666075c599b7,0.095687,0.000363,melhorou
4,5,IForest,ae79d360e696,0.095243,-0.000444,piorou


In [5]:
best_row = analysis_df.sort_values('objective_ap', ascending=False).iloc[0]
model_summary = (
    analysis_df
    .groupby('model', as_index=False)
    .agg(
        n_combos=('combo_id', 'nunique'),
        ap_best=('objective_ap', 'max'),
        ap_mediana=('objective_ap', 'median'),
        ap_media=('objective_ap', 'mean'),
        pior_ap=('objective_ap', 'min'),
        novos_recordes=('is_new_best_global', 'sum')
    )
    .sort_values('ap_best', ascending=False)
    .reset_index(drop=True)
)

print('Melhor combinacao global:')
print({
    'model': best_row['model'],
    'combo_id': best_row['combo_id'],
    'objective_ap': float(best_row['objective_ap']),
    'job_index': int(best_row['job_index'])
})
print()
print('Resumo por modelo:')
model_summary

Melhor combinacao global:
{'model': 'AutoEncoder', 'combo_id': 'eb95f118b8a5', 'objective_ap': 0.1349318799357655, 'job_index': 287}

Resumo por modelo:


,model,n_combos,ap_best,ap_mediana,ap_media,pior_ap,novos_recordes
0,AutoEncoder,144,0.134932,0.129883,0.127770,0.104551,3
1,INNE,15,0.100199,0.096236,0.095794,0.091091,1
2,IForest,90,0.099661,0.096341,0.096433,0.093393,8
3,CBLOF,30,0.086082,0.085583,0.085647,0.085091,0
4,LOF,20,0.083934,0.079090,0.079511,0.075789,0
5,LODA,20,0.080494,0.071921,0.073246,0.067161,0
6,HBOS,80,0.080396,0.077314,0.077195,0.075129,0


In [6]:
cols = [
    'job_index', 'model', 'combo_id', 'objective_ap',
    'delta_vs_prev_global', 'delta_vs_best_before',
    'delta_vs_prev_model', 'delta_vs_best_model_before',
    'trend_global', 'trend_model', 'total_job_time_s'
]

top_melhoras = analysis_df.sort_values('delta_vs_prev_global', ascending=False)[cols].head(20)
top_pioras = analysis_df.sort_values('delta_vs_prev_global', ascending=True)[cols].head(20)

def color_delta(val):
    try:
        if val > 0:
            return 'background-color: #c6f6d5; color: #22543d;'
        if val < 0:
            return 'background-color: #fed7d7; color: #742a2a;'
    except Exception:
        pass
    return ''

display(top_melhoras.style.applymap(color_delta, subset=['delta_vs_prev_global', 'delta_vs_prev_model', 'delta_vs_best_before', 'delta_vs_best_model_before']))
display(top_pioras.style.applymap(color_delta, subset=['delta_vs_prev_global', 'delta_vs_prev_model', 'delta_vs_best_before', 'delta_vs_best_model_before']))

/tmp/ipykernel_3248/888395026.py:21: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(top_melhoras.style.applymap(color_delta, subset=['delta_vs_prev_global', 'delta_vs_prev_model', 'delta_vs_best_before', 'delta_vs_best_model_before']))


,job_index,model,combo_id,objective_ap,delta_vs_prev_global,delta_vs_best_before,delta_vs_prev_model,delta_vs_best_model_before,trend_global,trend_model,total_job_time_s
255,282,AutoEncoder,e625f24a3b24,0.132077,0.048143,0.031878,0.000000,0.000000,melhorou,estavel,731.282400
327,354,AutoEncoder,a2f5e8572403,0.131312,0.026761,-0.003620,0.026761,-0.003620,melhorou,melhorou,1883.903000
283,310,AutoEncoder,acbc2614356a,0.133801,0.017078,-0.001131,0.017078,-0.001131,melhorou,melhorou,1090.088500
271,298,AutoEncoder,858207dc3177,0.133740,0.016702,-0.001192,0.016702,-0.001192,melhorou,melhorou,897.735300
287,314,AutoEncoder,6fff6045c619,0.132963,0.015579,-0.001969,0.015579,-0.001969,melhorou,melhorou,672.962700
373,370,AutoEncoder,389a6191e804,0.131524,0.015535,-0.003408,0.015535,-0.003408,melhorou,melhorou,2183.828100
374,370,AutoEncoder,389a6191e804,0.131524,0.015535,-0.003408,0.015535,-0.003408,melhorou,melhorou,2183.828100
295,322,AutoEncoder,479c3e6e6ea5,0.132608,0.014924,-0.002324,0.014924,-0.002324,melhorou,melhorou,1782.999700
422,382,AutoEncoder,7425700bdce4,0.130538,0.014353,-0.004394,0.014353,-0.004394,melhorou,melhorou,643.663400
421,382,AutoEncoder,7425700bdce4,0.130538,0.014353,-0.004394,0.014353,-0.004394,melhorou,melhorou,643.663400


/tmp/ipykernel_3248/888395026.py:22: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(top_pioras.style.applymap(color_delta, subset=['delta_vs_prev_global', 'delta_vs_prev_model', 'delta_vs_best_before', 'delta_vs_best_model_before']))


,job_index,model,combo_id,objective_ap,delta_vs_prev_global,delta_vs_best_before,delta_vs_prev_model,delta_vs_best_model_before,trend_global,trend_model,total_job_time_s
90,91,LODA,d02f484b1ba7,0.068280,-0.029116,-0.031382,0.000000,0.000000,piorou,estavel,1.237600
326,353,AutoEncoder,50671a1dff0c,0.104551,-0.025909,-0.030381,-0.025909,-0.030381,piorou,piorou,479.926200
274,301,AutoEncoder,cca8988eea5a,0.117489,-0.016633,-0.017442,-0.016633,-0.017442,piorou,piorou,2046.831400
235,262,LOF,2ede952a691c,0.075789,-0.016630,-0.024410,0.000000,0.000000,piorou,estavel,2921.727600
286,313,AutoEncoder,ac9f5fcb9c45,0.117383,-0.016118,-0.017549,-0.016118,-0.017549,piorou,piorou,1084.970800
282,309,AutoEncoder,1b58f0f6228a,0.116723,-0.015010,-0.018209,-0.015010,-0.018209,piorou,piorou,1914.955700
298,325,AutoEncoder,98b3c55a9bd9,0.117382,-0.015000,-0.017550,-0.015000,-0.017550,piorou,piorou,4143.103100
270,297,AutoEncoder,ba656c910e58,0.117038,-0.014891,-0.017894,-0.014891,-0.017894,piorou,piorou,3048.928200
258,285,AutoEncoder,9f0bc199a3c3,0.117376,-0.014386,-0.016109,-0.014386,-0.016109,piorou,piorou,1445.503500
302,329,AutoEncoder,f1565df17359,0.117536,-0.014288,-0.017396,-0.014288,-0.017396,piorou,piorou,3168.396800


In [7]:
# Tabela por modelo com esquema de cores de melhora/piora
cols_modelo = [
    'job_index', 'combo_id', 'objective_ap',
    'delta_vs_prev_global', 'delta_vs_prev_model',
    'delta_vs_best_before', 'delta_vs_best_model_before',
    'trend_global', 'trend_model', 'total_job_time_s'
 ]
cols_modelo = [c for c in cols_modelo if c in analysis_df.columns]

def color_delta_modelo(val):
    try:
        if val > 0:
            return 'background-color: #c6f6d5; color: #22543d;'
        if val < 0:
            return 'background-color: #fed7d7; color: #742a2a;'
    except Exception:
        pass
    return ''

TOP_N_POR_MODELO = 20
for model_name in sorted(analysis_df['model'].dropna().unique()):
    print(f'\n### Modelo: {model_name}')
    df_model = (
        analysis_df[analysis_df['model'] == model_name]
        .sort_values('job_index', ascending=True)
        .head(TOP_N_POR_MODELO)
        .reset_index(drop=True)
    )

    styled = df_model[cols_modelo].style.applymap(
        color_delta_modelo,
        subset=[c for c in [
            'delta_vs_prev_global',
            'delta_vs_prev_model',
            'delta_vs_best_before',
            'delta_vs_best_model_before'
        ] if c in cols_modelo]
    )
    display(styled)


### Modelo: AutoEncoder


/tmp/ipykernel_3248/3236835971.py:30: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = df_model[cols_modelo].style.applymap(


,job_index,combo_id,objective_ap,delta_vs_prev_global,delta_vs_prev_model,delta_vs_best_before,delta_vs_best_model_before,trend_global,trend_model,total_job_time_s
0,282,e625f24a3b24,0.132077,0.048143,0.000000,0.031878,0.000000,melhorou,estavel,731.282400
1,283,124569b9fe8a,0.133485,0.001408,0.001408,0.001408,0.001408,melhorou,melhorou,746.033500
2,284,73a04d15eec9,0.131762,-0.001723,-0.001723,-0.001723,-0.001723,piorou,piorou,958.038700
3,285,9f0bc199a3c3,0.117376,-0.014386,-0.014386,-0.016109,-0.016109,piorou,piorou,1445.503500
4,286,1b60bb177b05,0.130487,0.013111,0.013111,-0.002998,-0.002998,melhorou,melhorou,442.372300
5,287,eb95f118b8a5,0.134932,0.004445,0.004445,0.001447,0.001447,melhorou,melhorou,446.632300
6,288,15085e24afc1,0.133366,-0.001566,-0.001566,-0.001566,-0.001566,piorou,piorou,480.604500
7,289,e41e07601410,0.124237,-0.009129,-0.009129,-0.010695,-0.010695,piorou,piorou,992.014600
8,290,bd1f1789fdd1,0.129343,0.005106,0.005106,-0.005589,-0.005589,melhorou,melhorou,299.726100
9,291,cebc46f95f2b,0.133534,0.004191,0.004191,-0.001398,-0.001398,melhorou,melhorou,298.093300



### Modelo: CBLOF


/tmp/ipykernel_3248/3236835971.py:30: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = df_model[cols_modelo].style.applymap(


...

/tmp/ipykernel_3248/3236835971.py:30: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = df_model[cols_modelo].style.applymap(


,job_index,combo_id,objective_ap,delta_vs_prev_global,delta_vs_prev_model,delta_vs_best_before,delta_vs_best_model_before,trend_global,trend_model,total_job_time_s
0,91,d02f484b1ba7,0.068280,-0.029116,0.000000,-0.031382,0.000000,piorou,estavel,1.237600
1,92,56cf452ed656,0.067161,-0.001118,-0.001118,-0.032500,-0.001118,piorou,piorou,2.245600
2,93,0eec35339dc9,0.069411,0.002249,0.002249,-0.030250,0.001131,melhorou,melhorou,3.827800
3,94,2c3ec199d44f,0.072722,0.003311,0.003311,-0.026939,0.003311,melhorou,melhorou,8.817200
4,95,a8e15bd44f0a,0.070687,-0.002035,-0.002035,-0.028974,-0.002035,piorou,piorou,1.247200
5,96,7a4216e50e66,0.069496,-0.001191,-0.001191,-0.030165,-0.003226,piorou,piorou,2.244400
6,97,2f3bd5a2efe8,0.069906,0.000410,0.000410,-0.029755,-0.002816,melhorou,melhorou,3.877300
7,98,490bad69c646,0.071898,0.001992,0.001992,-0.027763,-0.000824,melhorou,melhorou,8.645700
8,99,97a2bedd94eb,0.069570,-0.002328,-0.002328,-0.030091,-0.003152,piorou,piorou,1.228500
9,100,d3c8fdc0a158,0.074068,0.004498,0.004498,-0.025594,0.001345,melhorou,melhorou,2.237400



### Modelo: LOF


/tmp/ipykernel_3248/3236835971.py:30: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled = df_model[cols_modelo].style.applymap(


,job_index,combo_id,objective_ap,delta_vs_prev_global,delta_vs_prev_model,delta_vs_best_before,delta_vs_best_model_before,trend_global,trend_model,total_job_time_s
0,262,2ede952a691c,0.075789,-0.016630,0.000000,-0.024410,0.000000,piorou,estavel,2921.727600
1,263,ba129e253b78,0.075789,0.000000,0.000000,-0.024410,0.000000,estavel,estavel,2942.166300
2,264,137f50c5594d,0.075789,0.000000,0.000000,-0.024410,0.000000,estavel,estavel,2923.627600
3,265,5f7f0fb645b2,0.075789,0.000000,0.000000,-0.024410,0.000000,estavel,estavel,3373.287300
4,266,3997d78bf49d,0.077016,0.001227,0.001227,-0.023183,0.001227,melhorou,melhorou,3364.537700
5,267,ecf7b6d2e606,0.077016,0.000000,0.000000,-0.023183,0.000000,estavel,estavel,3372.993400
6,268,9de932565ce3,0.077016,0.000000,0.000000,-0.023183,0.000000,estavel,estavel,3360.465400
7,269,9f266b52483c,0.077016,0.000000,0.000000,-0.023183,0.000000,estavel,estavel,3357.760400
8,270,1d1d5d7103a0,0.079090,0.002073,0.002073,-0.021110,0.002073,melhorou,melhorou,3359.839700
9,271,c2ff86d9d72d,0.079090,0.000000,0.000000,-0.021110,0.000000,estavel,estavel,3354.911300


In [8]:
if HAS_PLOTLY:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=analysis_df['job_index'],
        y=analysis_df['objective_ap'],
        mode='markers',
        marker=dict(size=8),
        text=analysis_df['model'] + ' | ' + analysis_df['combo_id'].astype(str),
        name='AP da combinacao'
    ))
    fig.add_trace(go.Scatter(
        x=analysis_df['job_index'],
        y=analysis_df['best_so_far'],
        mode='lines',
        line=dict(width=3),
        name='Melhor AP acumulado'
    ))

    new_best = analysis_df[analysis_df['is_new_best_global']]
    fig.add_trace(go.Scatter(
        x=new_best['job_index'],
        y=new_best['objective_ap'],
        mode='markers',
        marker=dict(size=10, symbol='diamond'),
        name='Novo recorde global'
    ))

    fig.update_layout(
        title='Evolucao do Objective AP ao longo dos jobs',
        xaxis_title='job_index',
        yaxis_title='objective_ap',
        template='plotly_white',
        height=520
    )
    fig.show()
else:
    print('Plotly nao disponivel para este grafico.')

In [9]:
if HAS_PLOTLY:
    fig_box = px.box(
        analysis_df,
        x='model',
        y='objective_ap',
        points='all',
        color='model',
        title='Distribuicao do objective_ap por modelo',
        template='plotly_white'
    )
    fig_box.update_layout(showlegend=False, height=520)
    fig_box.show()

    fig_tradeoff = px.scatter(
        analysis_df,
        x='total_job_time_s',
        y='objective_ap',
        color='model',
        hover_data=['job_index', 'combo_id', 'ap_combined', 'ap_individual_mean'],
        title='Trade-off: tempo total do job vs objective_ap',
        template='plotly_white'
    )
    fig_tradeoff.update_layout(height=520)
    fig_tradeoff.show()
else:
    print('Plotly nao disponivel para os graficos de distribuicao e trade-off.')

In [10]:
def tabela_exploracao(df: pd.DataFrame, model='TODOS', top_n=30, apenas_melhoras=False, ordenar_por='objective_ap'):
    tmp = df.copy()
    if model != 'TODOS':
        tmp = tmp[tmp['model'] == model]

    if apenas_melhoras:
        tmp = tmp[tmp['delta_vs_prev_global'] > 0]

    if ordenar_por not in tmp.columns:
        ordenar_por = 'objective_ap'

    tmp = tmp.sort_values(ordenar_por, ascending=False).head(top_n)

    show_cols = [
        'job_index', 'model', 'combo_id', 'objective_ap',
        'delta_vs_prev_global', 'delta_vs_prev_model',
        'ap_combined', 'ap_individual_mean', 'total_job_time_s',
        'params_json'
    ]
    show_cols = [c for c in show_cols if c in tmp.columns]
    return tmp[show_cols].reset_index(drop=True)

if HAS_WIDGETS:
    model_opts = ['TODOS'] + sorted(analysis_df['model'].dropna().unique().tolist())
    dd_model = widgets.Dropdown(options=model_opts, value='TODOS', description='Modelo:')
    sl_top = widgets.IntSlider(value=30, min=5, max=100, step=5, description='Top N:')
    ck_gain = widgets.Checkbox(value=False, description='Apenas melhoras')
    dd_sort = widgets.Dropdown(options=['objective_ap', 'delta_vs_prev_global', 'delta_vs_prev_model', 'total_job_time_s'], value='objective_ap', description='Ordenar:')

    out = widgets.interactive_output(
        lambda model, top_n, apenas_melhoras, ordenar_por: display(tabela_exploracao(analysis_df, model, top_n, apenas_melhoras, ordenar_por)),
        {
            'model': dd_model,
            'top_n': sl_top,
            'apenas_melhoras': ck_gain,
            'ordenar_por': dd_sort
        }
    )

    display(widgets.VBox([widgets.HBox([dd_model, dd_sort]), widgets.HBox([sl_top, ck_gain]), out]))
else:
    display(tabela_exploracao(analysis_df, model='TODOS', top_n=30, apenas_melhoras=False, ordenar_por='objective_ap'))

## Leitura rapida dos resultados

- Se `delta_vs_prev_global > 0`, a combinacao melhorou em relacao ao job anterior.
- Se `delta_vs_best_before > 0`, ela bateu o melhor resultado global anterior.
- Use os boxplots para comparar estabilidade por modelo.
- Use o scatter de trade-off para verificar ganho de AP versus custo de tempo.